# Feature Extraction Test Notebook

This notebook tests TF-IDF feature extraction and resume-job matching using cosine similarity.

In [28]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'tests' else Path.cwd()
src_path = project_root / 'src'
dataset_dir = project_root / 'Dataset'
train_csv = dataset_dir / 'train.csv'
test_csv = dataset_dir / 'test.csv'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from data_loader import load_train_test_data
from preprocessing import preprocess_dataframe
from feature_extraction import (
    create_tfidf_vectorizer,
    fit_tfidf,
    transform_tfidf,
    extract_train_test_features,
    rank_resumes_by_job_description,
)

print('Project root :', project_root)
print('Train exists :', train_csv.exists())
print('Test exists  :', test_csv.exists())

Project root : c:\Users\POOJA\OneDrive\Desktop\project
Train exists : True
Test exists  : True


## 1) Load and preprocess train/test data

In [29]:
train_df, test_df = load_train_test_data(dataset_dir)

train_processed = preprocess_dataframe(train_df, text_column='text')
test_processed = preprocess_dataframe(test_df, text_column='text')

print('Train shape:', train_processed.shape)
print('Test shape :', test_processed.shape)
display(train_processed.head(2))

Train shape: (1986, 2)
Test shape : (497, 2)


,text,label
0,engin consult profession summari deliv valu pr...,CONSULTANT
1,carpent summari carpent foreman posit effect u...,CONSTRUCTION


## 2) Test TF-IDF fit/transform

In [30]:
vectorizer = create_tfidf_vectorizer(max_features=3000, ngram_range=(1, 2), min_df=1)
vectorizer, x_train = fit_tfidf(train_processed['text'], vectorizer)
x_test = transform_tfidf(vectorizer, test_processed['text'])

print('x_train shape:', x_train.shape)
print('x_test shape :', x_test.shape)
print('Vocabulary size:', len(vectorizer.vocabulary_))

x_train shape: (1986, 3000)
x_test shape : (497, 3000)
Vocabulary size: 3000


In [31]:
sample_resume = train_processed['text'].iloc[0]
sample_vector = vectorizer.transform([sample_resume])
feature_names = vectorizer.get_feature_names_out()

sample_vector_df = pd.DataFrame({
    'term': feature_names,
    'tfidf_score': sample_vector.toarray().ravel(),
})

sample_vector_df = sample_vector_df[sample_vector_df['tfidf_score'] > 0].sort_values(
    by='tfidf_score',
    ascending=False,
).reset_index(drop=True)

print('Fitted TF-IDF vectorizer:')
print(vectorizer)
print()
print('Vocabulary size:', len(vectorizer.vocabulary_))
print('Sample vector shape:', sample_vector.shape)
print('Non-zero entries in sample vector:', sample_vector.nnz)
print('TF-IDF matrix shape from train data:', x_train.shape)
print()
print('Original Resume Sample:')
print(sample_resume)
print()
print('Top non-zero TF-IDF terms for this sample:')
display(sample_vector_df.head(20))

print('Ranking note: higher TF-IDF score means the term is more important in this resume sample.')

Fitted TF-IDF vectorizer:
TfidfVectorizer(max_features=3000, ngram_range=(1, 2), sublinear_tf=True)

Vocabulary size: 3000
Sample vector shape: (1, 3000)
Non-zero entries in sample vector: 292
TF-IDF matrix shape from train data: (1986, 3000)

Original Resume Sample:
engin consult profession summari deliv valu profession posit oil ga industri util attribut uniqu skillset long stand track record outperform manag goal mileston reduc time cost minim nonproduct time incorpor analyt creativ skill visual idea solut proactiv avoid problem depth understand mechan tool util optim function econom safeti experi margin product play lower predict cost key driver meticul invoic ensur cost accur agre develop indepth analyt mechan problem solv skill year field experi comprehens discuss vendor learninglisten experi wealth knowledg timelin workflow cost effect troubleshoot seamlessli integr field offic personnel unifi team righand experi field engin experi continu consid hse regulatori facet implement u

,term,tfidf_score
0,drill,0.294028
1,surfac,0.161912
2,lectur,0.158157
3,well,0.151363
4,engin,0.130686
5,play,0.125719
6,confirm,0.124685
7,pump,0.115821
8,sep,0.112906
9,bid,0.112337


Ranking note: higher TF-IDF score means the term is more important in this resume sample.


## 3) Resume ranking: vectorizer, matrix, and similarity score
This section shows:
- the fitted TF-IDF vectorizer
- the sparse TF-IDF matrix shape
- the non-zero TF-IDF terms for one resume sample
- the ranking score used for resume-job matching

## 3) Test one-call train/test feature extraction

In [32]:
vec2, x_train2, x_test2 = extract_train_test_features(
    train_processed,
    test_processed,
    text_column='text',
    max_features=4000,
    ngram_range=(1, 2),
    min_df=1,
)

summary_df = pd.DataFrame({
    'Matrix': ['x_train2', 'x_test2'],
    'Rows': [x_train2.shape[0], x_test2.shape[0]],
    'Columns': [x_train2.shape[1], x_test2.shape[1]],
})
summary_df
print(summary_df.iloc[0])

Matrix     x_train2
Rows           1986
Columns        4000
Name: 0, dtype: object


## 4) Resume ranking against a job description

In [37]:
sample_resumes = train_processed['text'].head(8).tolist()
job_description = 'Looking for data scientist with python machine learning and statistics skills .'

ranking_df = rank_resumes_by_job_description(job_description, sample_resumes, top_k=5)
ranking_df

,resume_index,similarity_score,resume_text
0,3,0.031879,sou chef work experi sou chef jul compani ï¼​ ...
1,2,0.031080,digit market specialist summari digit market p...
2,4,0.028790,donor advoc profession summari organ professio...
3,0,0.024806,engin consult profession summari deliv valu pr...
4,5,0.015347,hr manag summari human resourc manag practic u...


## 5) JD vector, resume matrix, and product details

In [38]:
from sklearn.metrics.pairwise import cosine_similarity

resume_texts = train_processed['text'].head(5).tolist()
resume_labels = [f'Resume {i+1}' for i in range(len(resume_texts))]

job_vector = vectorizer.transform([job_description])
resume_matrix = vectorizer.transform(resume_texts)
feature_names = vectorizer.get_feature_names_out()

job_dense = job_vector.toarray().ravel()
resume_dense = resume_matrix.toarray()

summary_shapes_df = pd.DataFrame({
    'Object': ['Job Description Vector', 'Resume Matrix'],
    'Shape': [job_vector.shape, resume_matrix.shape],
})

job_vector_df = pd.DataFrame({
    'term': feature_names[job_dense > 0],
    'jd_tfidf': job_dense[job_dense > 0],
}).sort_values(by='jd_tfidf', ascending=False).reset_index(drop=True)

ranking_table = pd.DataFrame({
    'resume_label': resume_labels,
    'similarity_score': cosine_similarity(job_vector, resume_matrix).flatten(),
    'resume_text': resume_texts,
}).sort_values(by='similarity_score', ascending=False).reset_index(drop=True)

all_products_rows = []
for idx, resume_label in enumerate(resume_labels):
    product_vector = resume_dense[idx] * job_dense
    nonzero_mask = product_vector > 0
    if nonzero_mask.any():
        product_df = pd.DataFrame({
            'resume_label': resume_label,
            'resume_text': resume_texts[idx],
            'term': feature_names[nonzero_mask],
            'jd_tfidf': job_dense[nonzero_mask],
            'resume_tfidf': resume_dense[idx][nonzero_mask],
            'product': product_vector[nonzero_mask],
        }).sort_values(by='product', ascending=False)
        all_products_rows.append(product_df)

all_products_df = pd.concat(all_products_rows, ignore_index=True) if all_products_rows else pd.DataFrame()

print('TF-IDF vectorizer used for ranking:')
print(vectorizer)
print()
display(summary_shapes_df)
print('Job Description (non-zero TF-IDF terms):')
display(job_vector_df.head(25))
print('Resumes reordered by cosine similarity to the JD:')
display(ranking_table)
print('Term-by-term products (JD TF-IDF x Resume TF-IDF), reordered by product value:')
display(all_products_df.head(50))

TF-IDF vectorizer used for ranking:
TfidfVectorizer(max_features=3000, ngram_range=(1, 2), sublinear_tf=True)



,Object,Shape
0,Job Description Vector,"(1, 3000)"
1,Resume Matrix,"(5, 3000)"


Job Description (non-zero TF-IDF terms):


,term,jd_tfidf
0,python,0.943930
1,data,0.330144


Resumes reordered by cosine similarity to the JD:


,resume_label,similarity_score,resume_text
0,Resume 4,0.013698,sou chef work experi sou chef jul compani ï¼​ ...
1,Resume 5,0.013460,donor advoc profession summari organ professio...
2,Resume 1,0.013220,engin consult profession summari deliv valu pr...
3,Resume 3,0.013050,digit market specialist summari digit market p...
4,Resume 2,0.000000,carpent summari carpent foreman posit effect u...


Term-by-term products (JD TF-IDF x Resume TF-IDF), reordered by product value:


,resume_label,resume_text,term,jd_tfidf,resume_tfidf,product
0,Resume 1,engin consult profession summari deliv valu pr...,data,0.330144,0.040042,0.013220
1,Resume 3,digit market specialist summari digit market p...,data,0.330144,0.039527,0.013050
2,Resume 4,sou chef work experi sou chef jul compani ï¼​ ...,data,0.330144,0.041490,0.013698
3,Resume 5,donor advoc profession summari organ professio...,data,0.330144,0.040770,0.013460
